# Where does the money actually get spent? (contract cross-check via makegov)

The [White House account notebook](white_house_account.ipynb) showed, from OMB's budget data, that the "Repair and Restoration" account (EOP `011-X-0109`) ballooned to \$323M — almost all reimbursable / collected funds. But OMB's apportionment and execution data stops at the *account*. It doesn't tell you what got contracted, or to whom.

To see that, you have to cross to the **contract** record — FPDS — which this notebook queries through [makegov](https://www.makegov.com/)'s Tango API, and check the linkage in [USAspending](https://www.usaspending.gov/).

> **Note on access.** Unlike the free, CC0 BlazingStar data used elsewhere in this repo, **makegov/Tango is a commercial API** — you need an account and an API key. Set it as `TANGO_API_KEY`. (USAspending, used at the end, is free and open.) This notebook won't run without the key, and it's intentionally left out of the repo's CI for that reason.

In [ ]:
%pip install -q tango-python requests pandas matplotlib

In [ ]:
import os

import pandas as pd
import requests
from tango import TangoClient

api_key = os.environ.get("TANGO_API_KEY")
assert api_key, "Set TANGO_API_KEY — a makegov API key (https://www.makegov.com/). This notebook needs it to query FPDS."
client = TangoClient(api_key=api_key)

## 1. The White House facilities contracts you *can* see

The repair-and-restoration work is contracted out through GSA (which buys for the White House complex). Pull the EOP-funded, GSA-awarded contracts since FY2025 — these are the visible White House facilities jobs.

In [ ]:
resp = client.list_contracts(
    funding_agency="Executive Office of the President",
    awarding_agency="General Services Administration",
    fiscal_year_gte=2025,
    order="desc", sort="total_contract_value", limit=50,
)
rows = [{
    "award_date": str(r.get("award_date"))[:10],
    "piid": r.get("piid"),
    "value": float(r.get("total_contract_value") or 0),
    "recipient": (r.get("recipient") or {}).get("display_name"),
    "description": (r.get("description") or "")[:50],
} for r in resp.results]
df = pd.DataFrame(rows)
visible_total = df["value"].sum()
print(f"contracts: {resp.count}   total contract value: ${visible_total:,.0f}")
print("Compare (SF-133, through May): ~$384M in budgetary resources, ~$82M actually spent (outlays).")
print(f"So visible contracts (~${visible_total/1e6:.1f}M) are a small fraction of the money.")
df

## 2. Who awards them, and who funds them

The PIIDs start with **47P** — that prefix is **GSA** (the Public Buildings Service does the contracting for the White House complex). So these are *awarded by GSA* and *funded by EOP*. Confirm one in USAspending — the $635K bathroom renovation at 712 Jackson Place (part of the White House complex).

In [ ]:
PIID = "47PH5426F0109"   # BATHROOM RENOVATION AT 712 JACKSON PLACE
S = requests.Session()
S.headers.update({"User-Agent": "blazingstar-data-demo"})

r = S.post("https://api.usaspending.gov/api/v2/search/spending_by_award/",
           json={"filters": {"keywords": [PIID], "award_type_codes": ["A", "B", "C", "D"]},
                 "fields": ["Award ID", "Recipient Name", "Awarding Agency", "Funding Agency",
                            "generated_internal_id"], "limit": 5}, timeout=60)
award = r.json()["results"][0]
print("Award:    ", award["Award ID"])
print("Recipient:", award["Recipient Name"])
print("Awarded by:", award["Awarding Agency"], " | Funded by:", award["Funding Agency"])

# Account-level (File C) linkage — which Treasury account actually paid?
gid = award["generated_internal_id"]
f = S.post("https://api.usaspending.gov/api/v2/awards/funding/",
           json={"award_id": gid, "limit": 10, "page": 1,
                 "sort": "reporting_fiscal_date", "order": "desc"}, timeout=60)
funding = f.json().get("results", [])
print("\nTreasury-account funding records linked to this award:", len(funding))
for rec in funding:
    print("  ", rec.get("federal_account"), rec.get("treasury_account_symbol"), rec.get("funding_agency_name"))

## 3. The funnel: ~$82M spent, ~$2M visible as contracts

Put the budget side and the contract side on one axis. A note on terms: **obligated** = money *committed* (a signed contract/order), **outlays** = cash *actually paid out* — the latter is "spent." These are the account-family totals from the SF-133, through May.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
plt.rcParams["text.parse_math"] = False
ACCENT = "#69539E"

resources = 384.3e6          # total budgetary resources through May (SF-133, account 0109 family)
spent = 82.3e6               # gross outlays — cash actually spent — through May (SF-133)
contracts = float(visible_total)   # from the makegov query above

labels = ["Budgetary\nresources", "Spent\n(outlays)", "Visible as\ncontracts"]
vals = [resources, spent, contracts]
ypos = [2, 1, 0]

fig, ax = plt.subplots(figsize=(10, 5.2))
fig.subplots_adjust(top=0.76, bottom=0.14, left=0.14, right=0.96)
ax.barh(ypos, vals, color=ACCENT, height=0.6)
for y, v in zip(ypos, vals):
    extra = f"  ({v / resources * 100:.1f}%)" if y < 2 else ""
    ax.text(v + resources * 0.012, y, f"${v / 1e6:,.0f}M{extra}", va="center", ha="left",
            fontsize=11, fontweight="bold", color="#222")
ax.set_yticks(ypos); ax.set_yticklabels(labels, fontsize=10.5)
ax.set_xlim(0, resources * 1.12)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v / 1e6:,.0f}M"))
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(left=False)
ax.grid(axis="x", color="#ededed", lw=0.8); ax.set_axisbelow(True)

fig.text(0.045, 0.93, "White House Repair and Restoration: most of the money isn't visible as contracts",
         fontsize=13.5, fontweight="bold", color=ACCENT, ha="left", va="top")
fig.text(0.045, 0.85,
         "Account 011-0109 (all sub-accounts), through May FY2026. Of $384M in budgetary resources and ~$82M\n"
         "actually spent (outlays), only ~$2M shows up as federal contracts — and even those can't be tied to the account.",
         fontsize=10, color="#444", ha="left", va="top", linespacing=1.5)
fig.text(0.045, 0.02,
         "Sources: OMB SF-133 via BlazingStar (CC0); contracts via makegov (FPDS) + USAspending "
         "(EOP-funded, GSA-awarded, FY2025+).",
         fontsize=7.5, color="#999", ha="left", va="bottom")
fig.savefig("whitehouse_contracts_funnel.png", dpi=200, facecolor="white")
plt.show()

## What this shows

- **It isn't nothing — but it's a sliver.** You *can* see real White House facilities contracts: ~\$2M across ~20 GSA-awarded, EOP-funded jobs since FY2025 (a \$635K bathroom renovation at 712 Jackson Place, a \$418K feasibility study, carpet/flooring, HVAC, a \$22K West Wing chandelier install). But that ~\$2M is a small fraction of the ~\$82M the account *actually spent* (outlays) by May, and tiny against its \$384M in budgetary resources. **Where the rest of the money goes doesn't show up as trackable federal contracts.**
- **And you can't tie even these to the specific account.** The award's account-level (File C) linkage in USAspending is empty, and FPDS carries no Treasury account at all. So the \$323M "Repair and Restoration" apportionment and these visible contracts **can't be rigorously connected in public data**.
- **The contracting runs through GSA**, which fits the reimbursable / Economy-Act funding on the budget side: EOP's account takes money in, GSA does the buying.
- A keyword search for `ballroom` returns **no** federal contracts by that description.

**Bottom line:** the budget side (BlazingStar/OMB) shows the money arriving and being obligated; the contract side (FPDS/USAspending) shows only a couple million in small facilities jobs. Most of the money — and the join between the two halves — isn't visible in public data.